<a href="https://colab.research.google.com/github/hmmnyamminji/DL/blob/main/day15_practice3_%EA%B0%90%EC%84%B1%EB%B6%84%EB%A5%98_NSMC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# 네이버 영화 리뷰 감성 분류

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import urllib.request, os
from collections import Counter # Counter=단어 빈도 세기

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [13]:
# 셀 1. NSMC - 네이버 영화 리뷰
URL = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
urllib.request.urlretrieve(URL, "ratings_train.txt")
print("ratings_train.txt 다운로드 완료")

texts, labels = [], []

with open("ratings_train.txt", encoding="utf-8") as f:
  next(f) # 첫 줄 (레더 id document label)을 한 번 읽어 버림
  for line in f:
    parts = line.strip().split("\t")
    if len(parts) == 3 and parts[1]:
      texts.append(parts[1])
      labels.append(int(parts[2]))

print(f"전체 리뷰 {len(texts):,}개 | 긍정 {sum(labels):,} / 부정 {len(labels)-sum(labels):,}")
print("샘플:", texts[0], "->", "긍정" if labels[0] else "부정")

N = 30000
texts, labels = texts[:N], labels[:N] # 3만개만 슬라이싱


ratings_train.txt 다운로드 완료
전체 리뷰 149,995개 | 긍정 74,825 / 부정 75,170
샘플: 아 더빙.. 진짜 짜증나네요 목소리 -> 부정


In [15]:
# 셀 2. vocab - '빈도 상위'만
counter = Counter(tck for t in texts for tck in t.split()) # {단어: 등장횟수}
print(f"\n 고유 어절 수: {len(counter):,}개")
print("최다 빈도:", counter.most_common(5))

VOCAB_SIZE = 15000
vocab = {"<pad>": 0, "<unk>": 1}
for tck, _ in counter.most_common(VOCAB_SIZE - 2): # 특수토큰 2개 뺀 만큼
  vocab[tck] = len(vocab) # 번호 부여

MAX_LEN = 20
def encode(text):
  ids = [vocab.get(t, 1) for t in text.split()][:MAX_LEN] # 단어를 번호로 바꾸고, 너무 길면 자른다.
  return ids + [0] * (MAX_LEN - len(ids)) # 너무 짧으면 위를 0으로 채운다.

X = torch.tensor([encode(t) for t in texts])
y = torch.tensor(labels, dtype=torch.float32).reshape(-1,1)

# 학습/평가 분리
n_train = int(N * 0.9)
train_loader = DataLoader(TensorDataset(X[:n_train], y[:n_train]), batch_size=256, shuffle=True) # 앞 90%
X_test, y_test = X[n_train:].to(device), y[n_train:].to(device) # 뒤 10%


 고유 어절 수: 98,463개
최다 빈도: [('영화', 2241), ('너무', 1602), ('정말', 1566), ('진짜', 1191), ('이', 1021)]


In [18]:
# 셀 3. 모델 - 임베딩 + 평균  + MLP
class SentimentNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.emb = nn.Embedding(len(vocab), 64, padding_idx=0) # 임베딩 표(단어수X64차원벡터)
    self.fc = nn.Sequential(
        nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 1), nn.Sigmoid()) # 긍정/부정

  def forward(self, x): # x: (B, 20) 각 문장이 단어번호 20개
    mask = (x != 0).unsqueeze(-1).float() # (B, 20, 1) 1. 마스크 만들기: 어디가 진짜 단어이고 어디가 pad 인가
    vecs = self.emb(x) * mask # (B, 20, 64) 각 문장의 단어수 20x64차원 벡터  2. 단어 벡터를 꺼내고, pad 자리는 0으로 지우기
    sent = vecs.sum(1)/mask.sum(1).clamp(min=1) # (B,64) 벡터들의 합/진짜 단어 개수   3. '진짜 단어 개수'로만 나눠 평균, clamp(min=1)나누는 값이 최소1
    return self.fc(sent) # (B,64) 문장벡터 → 긍정 확률(B,1)

model = SentimentNet().to(device)
loss_fn = nn.BCELoss()
opt = torch.optim.Adam(model.parameters(), lr=0.002)
print(f"\n 모델 파라미터: {sum(p.numel() for p in model.parameters())/1e3:.0f}K"
      f" 임베딩 표가 {len(vocab)*64/1e3:.0f}K")


 모델 파라미터: 962K 임베딩 표가 960K
